In [2]:
import pandas as pd

#Load Datasets
df_domestic = pd.read_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/dedup_domestic_2025_09.parquet") # Update File name as needed

df_roamers = pd.read_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/dedup_roaming_2025_09.parquet") # Update File name as needed

In [5]:
len(df_domestic)

62689199

In [8]:
len(df_roamers)

1081350

## Fake Device Analysis

### 1. Domestic Subscribers

In [9]:
# Generate Subset of Fake Devices - Associated to Domestic Subscribers

# 1. Define the specific columns to check - if all these are null, we consider the device to be FAKE
device_cols = ['oem', 'brand', 'model', 'marketing_name', 'device_type']

# 2. Create a boolean mask where True means ALL specified columns are null.
fake_mask = df_domestic[device_cols].isna().all(axis=1)

# 3. Apply the mask to your  DataFrames to create the subsets
df_fake_domestic = df_domestic[fake_mask]

# 4. Now we drop the columns where records are Null due to devices being FAKE. 
df_fake_domestic.drop(columns=['oem', 'brand', 'model', 'marketing_name', 'device_type', 'os_family', 'os_version', 'sim_slots', 'has_2g', 'has_3g', 'has_4g', 'has_5g', 'year_released'], inplace=True)

# 5. We Replace tac with 'fake' to easily identify these records in the future.
df_fake_domestic['tac'] = 'fake'

# 6. Preview 
df_fake_domestic.head()

,first_seen,imei,imsi,last_seen,msisdn,tac,first_name,surname,id_type,id_number,prefix,gender,birth_year,age,district,mno
127,2025-09-01 00:34:09,22222222222222,641010279559509,2026-03-08 21:07:01,256754685197,fake,ESEZA,NAKANWAGI,NATIONAL_ID,CF93047105ET2G,075,Female,1993,33,KAYUNGA,AIRTEL
316,2025-09-04 14:50:00,35585532455125,641101929901293,2025-12-29 04:29:34,256760939554,fake,SYLIVIA,TEKYEKISA,NATIONAL_ID,CF88047108JL4C,076,Female,1988,38,KAYUNGA,MTN
343,2025-09-17 14:09:08,35289450073561,641101960598283,2026-02-11 21:32:05,256788440432,fake,ROSE,AKELLO,NATIONAL_ID,CF770711031RDF,078,Female,1977,49,AMURU,MTN
557,2025-09-05 07:47:12,22324514345555,641101949096707,2025-09-26 15:51:21,256766867064,fake,JAFAR,HASMANI,NATIONAL_ID,CM75033100Q2YJ,076,Male,1975,51,NEBBI,MTN
977,2025-09-04 16:45:27,35400565766676,641101922605631,2026-02-25 12:30:08,256760233757,fake,BETTY,NALUBEGA,NATIONAL_ID,CF8505210DQNLE,076,Female,1985,41,WAKISO,MTN


In [10]:
len(df_fake_domestic)

2709323

In [11]:
df_fake_domestic.head()

,first_seen,imei,imsi,last_seen,msisdn,tac,first_name,surname,id_type,id_number,prefix,gender,birth_year,age,district,mno
127,2025-09-01 00:34:09,22222222222222,641010279559509,2026-03-08 21:07:01,256754685197,fake,ESEZA,NAKANWAGI,NATIONAL_ID,CF93047105ET2G,075,Female,1993,33,KAYUNGA,AIRTEL
316,2025-09-04 14:50:00,35585532455125,641101929901293,2025-12-29 04:29:34,256760939554,fake,SYLIVIA,TEKYEKISA,NATIONAL_ID,CF88047108JL4C,076,Female,1988,38,KAYUNGA,MTN
343,2025-09-17 14:09:08,35289450073561,641101960598283,2026-02-11 21:32:05,256788440432,fake,ROSE,AKELLO,NATIONAL_ID,CF770711031RDF,078,Female,1977,49,AMURU,MTN
557,2025-09-05 07:47:12,22324514345555,641101949096707,2025-09-26 15:51:21,256766867064,fake,JAFAR,HASMANI,NATIONAL_ID,CM75033100Q2YJ,076,Male,1975,51,NEBBI,MTN
977,2025-09-04 16:45:27,35400565766676,641101922605631,2026-02-25 12:30:08,256760233757,fake,BETTY,NALUBEGA,NATIONAL_ID,CF8505210DQNLE,076,Female,1985,41,WAKISO,MTN


In [12]:
import pandas as pd
from IPython.display import display, HTML

# ==========================================
# 1. DATA PREPARATION
# ==========================================
# Convert 'first_seen' to datetime and extract the Year-Month period
df_fake_domestic['first_seen'] = pd.to_datetime(df_fake_domestic['first_seen'], errors='coerce')
df_fake_domestic['year_month'] = df_fake_domestic['first_seen'].dt.to_period('M')

# Recreate the age brackets (coercing errors so blanks become NaN)
if "age_bracket" not in df_fake_domestic.columns:
    bins   = [0, 17, 30, 40, 50, 60, 70, 100, 120]
    labels = ["<18", "18-30", "31-40", "41-50", "51-60", "61-70", "71-100", "100+"]
    numeric_age = pd.to_numeric(df_fake_domestic["age"], errors='coerce')
    df_fake_domestic["age_bracket"] = pd.cut(numeric_age, bins=bins, labels=labels, right=True)
    df_fake_domestic["age_bracket"] = pd.Categorical(df_fake_domestic["age_bracket"], categories=labels, ordered=True)

# Fill missing demographics with 'Unknown' so Pandas counts them rather than dropping them
mno_col = df_fake_domestic['mno'].fillna('Unknown MNO')
gender_col = df_fake_domestic['gender'].fillna('Unknown Gender')

# ==========================================
# 2. GENERATE TIME-SERIES REPORTS
# ==========================================

print("1. Total Fake Devices Detected per Month):")
print("="*70)
# value_counts gets the counts, sort_index puts them in chronological order
total_monthly = df_fake_domestic['year_month'].value_counts().sort_index().to_frame(name='Total Fake IMEIs')
total_monthly.loc['TOTAL'] = total_monthly.sum() # Add grand total row
display(total_monthly)

print("2. Fake Devices per MNO by Month:")
print("="*70)
# crosstab automatically builds the grid and calculates margins (row/col totals)
mno_monthly = pd.crosstab(
    index=df_fake_domestic['year_month'], 
    columns=mno_col, 
    margins=True, 
    margins_name="TOTAL"
)
display(mno_monthly)

print("3. Fake Devices per Gender by Month:")
print("="*70)
gender_monthly = pd.crosstab(
    index=df_fake_domestic['year_month'], 
    columns=gender_col, 
    margins=True, 
    margins_name="TOTAL"
)
display(gender_monthly)

print("4. Fake Devices per Age Group by Month:")
print("="*70)
# For the age bracket (which is categorical), we use dropna=False to show the 'NaN' column 
# representing records with no age data.
age_monthly = pd.crosstab(
    index=df_fake_domestic['year_month'], 
    columns=df_fake_domestic['age_bracket'], 
    dropna=False, 
    margins=True, 
    margins_name="TOTAL"
)
display(age_monthly)

print("5. Fake Devices per District of Origin by Month:")
print("="*70)
district_monthly = pd.crosstab(
    index=df_fake_domestic['year_month'], 
    columns=df_fake_domestic['district'], 
    margins=True, 
    margins_name="TOTAL"
)
display(district_monthly)

1. Total Fake Devices Detected per Month):


,Total Fake IMEIs
year_month,
2025-09,2709323
TOTAL,2709323


2. Fake Devices per MNO by Month:


mno,AIRTEL,HAMILTON,MTN,UNKNOWN,TOTAL
year_month,,,,,
2025-09,1287310,1,1422008,4,2709323
TOTAL,1287310,1,1422008,4,2709323


3. Fake Devices per Gender by Month:


gender,Female,Male,Undefined,Unknown Gender,TOTAL
year_month,,,,,
2025-09,1134565,1142404,21,432333,2709323
TOTAL,1134565,1142404,21,432333,2709323


4. Fake Devices per Age Group by Month:


age_bracket,<18,18-30,31-40,41-50,51-60,61-70,71-100,100+,TOTAL
year_month,,,,,,,,,
2025-09,153,788041,708837,417823,230370,91688,39999,0,2709323
TOTAL,153,788041,708837,417823,230370,91688,39999,0,2709323


5. Fake Devices per District of Origin by Month:


district,ABIM,ADJUMANI,AGAGO,ALEBTONG,AMOLATAR,AMUDAT,AMURIA,AMURU,APAC,ARUA,...,SHEEMA,SIRONKO,SOROTI,SSEMBABULE,TORORO,UNKNOWN,WAKISO,YUMBE,ZOMBO,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2025-09,3641,5215,8331,14460,6485,474,10491,5792,13564,36092,...,23202,25112,11973,18279,37088,7289,111363,14370,13905,2276990
TOTAL,3641,5215,8331,14460,6485,474,10491,5792,13564,36092,...,23202,25112,11973,18279,37088,7289,111363,14370,13905,2276990


### 2. Roaming Subscribers

In [13]:
# Generate Subset of Fake Devices - Associated to Roaming Subscribers

# 1. Define the specific columns to check - if all these are null, we consider the device to be FAKE
device_cols = ['oem', 'brand', 'model', 'marketing_name', 'device_type']

# 2. Create a boolean mask where True means ALL specified columns are null.
fake_mask = df_roamers[device_cols].isna().all(axis=1)

# 3. Apply the mask to your  DataFrames to create the subsets
df_fake_roamers = df_roamers[fake_mask]

# 4. Now we drop the columns where records are Null due to devices being FAKE. 
df_fake_roamers.drop(columns=['oem', 'brand', 'model', 'marketing_name', 'device_type', 'os_family', 'os_version', 'sim_slots', 'has_2g', 'has_3g', 'has_4g', 'has_5g', 'year_released'], inplace=True)

# 5. We Replace tac with 'fake' to easily identify these records in the future.
df_fake_roamers['tac'] = 'fake'

# 6. Preview 
df_fake_roamers.head()

,first_seen,imei,imsi,last_seen,msisdn,tac,mcc,country
1131180,2025-09-01 00:00:09,35199887876565,639035042000262,NaT,<NA>,fake,639,Kenya
8553799,2025-09-01 00:26:02,89009876544567,639035521535471,2026-01-26 15:36:21,254736345048,fake,639,Kenya
14224930,2025-09-01 00:29:02,35824081414346,639021391828612,2026-01-06 01:16:10,254714988653,fake,639,Kenya
7239597,2025-09-01 00:32:41,54931814476234,639035046619748,2026-02-18 16:16:54,,fake,639,Kenya
17285336,2025-09-01 00:42:26,35766554566469,635105021179063,NaT,<NA>,fake,635,Rwanda


In [14]:
len(df_fake_roamers)

31556

In [15]:
import pandas as pd
from IPython.display import display, HTML

# ==========================================
# 1. DATA PREPARATION
# ==========================================
# Convert 'first_seen' to datetime and extract the Year-Month period
df_fake_roamers['first_seen'] = pd.to_datetime(df_fake_roamers['first_seen'], errors='coerce')
df_fake_roamers['year_month'] = df_fake_roamers['first_seen'].dt.to_period('M')


# ==========================================
# 2. GENERATE TIME-SERIES REPORTS
# ==========================================

print("1. Total Fake Devices Detected per Month):")
print("="*70)
# value_counts gets the counts, sort_index puts them in chronological order
total_monthly = df_fake_roamers['year_month'].value_counts().sort_index().to_frame(name='Total Fake IMEIs')
total_monthly.loc['TOTAL'] = total_monthly.sum() # Add grand total row
display(total_monthly)


print("5. Fake Devices per Country of Origin by Month:")
print("="*70)
country_monthly = pd.crosstab(
    index=df_fake_roamers['year_month'], 
    columns=df_fake_roamers['country'], 
    margins=True, 
    margins_name="TOTAL"
)
display(country_monthly)

1. Total Fake Devices Detected per Month):


,Total Fake IMEIs
year_month,
2025-09,31556
TOTAL,31556


5. Fake Devices per Country of Origin by Month:


country,Austria,Bangladesh,Belgium,Botswana,Burundi,Cameroon,Canada,Central African Republic,Chad,China,...,Switzerland,Tanzania,Togo,Turkiye,United Arab Emirates,United Kingdom,Vietnam,Zambia,Zimbabwe,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2025-09,1,2,78,1,29,1,1,17,2,17,...,1,310,1,3,219,122,6,26,1,31556
TOTAL,1,2,78,1,29,1,1,17,2,17,...,1,310,1,3,219,122,6,26,1,31556


## Add PWDs Data Set and Enrich, all Cases

## Cloned Device Analysis

# Come to this Rubbish Later, and Generate something nice of it

In [8]:
import pandas as pd

# 1. Ensure your date column is a datetime object
# (Swap 'first_seen' for 'last_seen' if you prefer to group by their most recent activity)
df_domestic['first_seen'] = pd.to_datetime(df_domestic['first_seen'])

# 2. Create a Year-Month grouping column (e.g., '2025-08')
df_domestic['year_month'] = df_domestic['first_seen'].dt.to_period('M')

# 3. Sort the months chronologically so the report prints in order
months = df_domestic['year_month'].sort_values().unique()

# 4. Iterate through each month and generate the report
for month in months:
    # Extract just the data for this specific month
    monthly_df = df_domestic[df_domestic['year_month'] == month]
    
    print("="*60)
    print(f"REPORT FOR MONTH: {month}")
    print("-" * 60)
    
    # Total MSISDNs for the month
    print(f"Total Registered Numbers (MSISDNs): {len(monthly_df):,}")

    # Unique Subscribers for the month
    # value_counts() automatically drops NaNs, ensuring we only count valid IDs
    sims_per_id = monthly_df["id_number"].value_counts()
    print(f"Total Unique Subscribers (incl. Corporate): {len(sims_per_id):,}")
    print("-" * 60)

    # Create mutually exclusive buckets
    bins = [0, 1, 10, 100, float('inf')]
    labels = ["Exactly 1 SIM", "2 to 10 SIMs", "11 to 100 SIMs", "More than 100 SIMs"]

    # Group the counts into the bins
    distribution = pd.cut(sims_per_id, bins=bins, labels=labels).value_counts(sort=False)

    # Print the distribution table
    print("SIM Ownership Distribution:")
    for category, count in distribution.items():
        print(f" -> {category:<20}: {count:>12,} subscribers")

print("="*60)
print("END OF REPORT")

REPORT FOR MONTH: 2025-09
------------------------------------------------------------
Total Registered Numbers (MSISDNs): 76,342,848
Total Unique Subscribers (incl. Corporate): 13,846,173
------------------------------------------------------------
SIM Ownership Distribution:
 -> Exactly 1 SIM       :    2,630,559 subscribers
 -> 2 to 10 SIMs        :    9,869,607 subscribers
 -> 11 to 100 SIMs      :    1,345,350 subscribers
 -> More than 100 SIMs  :          657 subscribers
REPORT FOR MONTH: 2025-10
------------------------------------------------------------
Total Registered Numbers (MSISDNs): 42,577,174
Total Unique Subscribers (incl. Corporate): 8,890,446
------------------------------------------------------------
SIM Ownership Distribution:
 -> Exactly 1 SIM       :    2,326,774 subscribers
 -> 2 to 10 SIMs        :    5,880,810 subscribers
 -> 11 to 100 SIMs      :      682,264 subscribers
 -> More than 100 SIMs  :          598 subscribers
REPORT FOR MONTH: 2025-11
-----------

In [9]:
import pandas as pd
from IPython.display import display, HTML

# 1. Ensure the helper function is defined ONCE outside the loop
def generate_mno_summary(df, value_col, agg_method):
    """Generates a summary matrix with Row and Column totals."""
    summary = pd.pivot_table(
        df,
        index="mno",
        columns="id_type",
        values=value_col,
        aggfunc=agg_method,
        observed=False,
        fill_value=0
    )

    # Add Row Totals and Sort
    summary["Total"] = summary.sum(axis=1)
    summary = summary.sort_values("Total", ascending=False)

    # Add the Grand Total row at the bottom
    summary.loc["TOTAL"] = summary.sum(axis=0)

    return summary

# 2. Ensure your date column is ready (if you haven't already run this from the previous step)
# df_domestic['first_seen'] = pd.to_datetime(df_domestic['first_seen'])
# df_domestic['year_month'] = df_domestic['first_seen'].dt.to_period('M')

# 3. Sort the months chronologically
months = df_domestic['year_month'].sort_values().unique()

# 4. Iterate and generate styled tables for each month
for month in months:
    # Filter the dataset for the specific month
    monthly_df = df_domestic[df_domestic['year_month'] == month]
    
    # Print a nice HTML header for the notebook
    display(HTML(f"<hr><h2>REPORT FOR MONTH: {month}</h2>"))
    
    # --- Generate the underlying numeric tables ---
    # Note: passing 'monthly_df' instead of the full 'df'
    idtype_summary = generate_mno_summary(monthly_df, "id_number", "nunique")
    idtype_msisdn_summary = generate_mno_summary(monthly_df, "msisdn", "count")

    # --- Display them beautifully using Pandas Styler ---
    print(f"Unique Subscribers per MNO by ID Type ({month})")
    display(idtype_summary.style.format("{:,.0f}"))

    print(f"\nMSISDN Counts per MNO by ID Type ({month})")
    display(idtype_msisdn_summary.style.format("{:,.0f}"))

Unique Subscribers per MNO by ID Type (2025-09)


id_type,COMPANY_ID,EMPLOYEE_ID,NATIONAL_ID,PASSPORT,REFUGEE_ID,UNKNOWN,Total
mno,,,,,,,
MTN,0,"12,375","11,308,281","20,776","384,943",0,"11,726,375"
AIRTEL,"6,731",0,"7,932,328","6,221","71,307",0,"8,016,587"
HAMILTON,0,0,0,0,0,0,0
LYCA,0,0,0,0,0,0,0
TALKIO,0,0,0,0,0,0,0
UTCL,0,0,0,0,0,0,0
TOTAL,"6,731","12,375","19,240,609","26,997","456,250",0,"19,742,962"



MSISDN Counts per MNO by ID Type (2025-09)


id_type,COMPANY_ID,EMPLOYEE_ID,NATIONAL_ID,PASSPORT,REFUGEE_ID,UNKNOWN,Total
mno,,,,,,,
MTN,0,"342,054","32,517,564","31,696","1,119,212",0,"34,010,526"
AIRTEL,"249,731",0,"33,205,985","9,815","170,414",0,"33,635,945"
HAMILTON,0,0,0,0,0,0,0
LYCA,0,0,0,0,0,0,0
TALKIO,0,0,0,0,0,0,0
UTCL,0,0,0,0,0,0,0
TOTAL,"249,731","342,054","65,723,549","41,511","1,289,626",0,"67,646,471"


Unique Subscribers per MNO by ID Type (2025-10)


...

id_type,COMPANY_ID,EMPLOYEE_ID,NATIONAL_ID,PASSPORT,REFUGEE_ID,UNKNOWN,Total
mno,,,,,,,
AIRTEL,"130,646",0,"16,232,111","3,928","95,281",0,"16,461,966"
MTN,0,"36,090","5,823,687","11,243","222,309",0,"6,093,329"
HAMILTON,0,0,6,0,0,0,6
LYCA,0,0,0,0,0,0,0
TALKIO,0,0,0,0,0,0,0
UTCL,0,0,0,0,0,0,0
TOTAL,"130,646","36,090","22,055,804","15,171","317,590",0,"22,555,301"


Unique Subscribers per MNO by ID Type (2026-03)


id_type,COMPANY_ID,EMPLOYEE_ID,NATIONAL_ID,PASSPORT,REFUGEE_ID,UNKNOWN,Total
mno,,,,,,,
AIRTEL,"1,419",0,"1,626,130",486,"11,947",0,"1,639,982"
MTN,0,79,"34,165",105,"1,519",0,"35,868"
HAMILTON,0,0,0,0,0,0,0
LYCA,0,0,0,0,0,0,0
TALKIO,0,0,0,0,0,0,0
UTCL,0,0,0,0,0,0,0
TOTAL,"1,419",79,"1,660,295",591,"13,466",0,"1,675,850"



MSISDN Counts per MNO by ID Type (2026-03)


id_type,COMPANY_ID,EMPLOYEE_ID,NATIONAL_ID,PASSPORT,REFUGEE_ID,UNKNOWN,Total
mno,,,,,,,
AIRTEL,"23,616",0,"4,333,337",849,"25,743",0,"4,383,545"
MTN,0,804,"70,510",170,"3,115",0,"74,599"
HAMILTON,0,0,0,0,0,0,0
LYCA,0,0,0,0,0,0,0
TALKIO,0,0,0,0,0,0,0
UTCL,0,0,0,0,0,0,0
TOTAL,"23,616",804,"4,403,847","1,019","28,858",0,"4,458,144"


In [10]:
from IPython.display import display, HTML

# 1. Ensure we have the months from the datetime column
# (Assuming df_domestic['year_month'] was already created in the previous steps)
months = df_domestic['year_month'].sort_values().unique()

for month in months:
    # 2. Filter for BOTH the specific month AND only National IDs
    df_month_nid = df_domestic[(df_domestic['year_month'] == month) & (df_domestic['id_type'] == 'NATIONAL_ID')]
    
    # Skip months that might not have any NID registrations to avoid division by zero errors
    if len(df_month_nid) == 0:
        continue

    # Print an HTML header to separate the months clearly in your notebook
    display(HTML(f"<br><hr><h2>NID COMPLIANCE REPORT FOR MONTH: {month}</h2>"))
    
    print("="*70)
    
    # -------------------------------------------------
    # 1) GLOBAL NID METRICS
    # -------------------------------------------------
    total_msisdns = len(df_month_nid)
    unique_subs = df_month_nid["id_number"].nunique()

    print(f"Total MSISDNs Registered with NID:      {total_msisdns:>12,}")
    print(f"Total Unique Subscribers (NID):   {unique_subs:>12,}")
    print(f"Average SIMs per Person:                {total_msisdns / unique_subs:>12.2f}")
    print("-" * 70)

    # -------------------------------------------------
    # 2) CROSS-NETWORK BEHAVIOR (Multi-Homing)
    # -------------------------------------------------
    mno_per_sub = df_month_nid.groupby("id_number")["mno"].nunique()
    single_network_subs = (mno_per_sub == 1).sum()
    multi_network_subs = (mno_per_sub > 1).sum()

    print("NETWORK LOYALTY (Cross-Network Presence):")
    print(f" * Active on exactly 1 Network:        {single_network_subs:>12,} ({(single_network_subs/unique_subs)*100:.1f}%)")
    print(f" * Active on 2+ Networks:              {multi_network_subs:>12,} ({(multi_network_subs/unique_subs)*100:.1f}%)")
    print("-" * 70)

    # -------------------------------------------------
    # 3) STRICT COMPLIANCE: SIMs PER SINGLE NETWORK
    # -------------------------------------------------
    # Get the MAXIMUM number of SIMs a user holds on ANY single network
    sims_per_network = df_month_nid.groupby(["id_number", "mno"], observed=True).size()
    max_sims_single_mno = sims_per_network.groupby("id_number", observed=True).max()

    exactly_one_strict = (max_sims_single_mno == 1).sum()
    two_to_ten_strict = ((max_sims_single_mno > 1) & (max_sims_single_mno <= 10)).sum()
    ucc_violators = (max_sims_single_mno > 10).sum()

    print("COMPLIANCE AUDIT (SIMs held on a SINGLE Network):")
    print(f" * Exactly 1 SIM (Standard User):      {exactly_one_strict:>12,}")
    print(f" * 2 to 10 SIMs (Power User):          {two_to_ten_strict:>12,}")
    print(f" * >10 SIMs (UCC RULE VIOLATORS!):     {ucc_violators:>12,} ")
    print("-"*70)

    # -------------------------------------------------
    # 4) CROSS NETWORK COMPLIANCE: SIMs ACROSS ALL NETWORKS
    # -------------------------------------------------
    # Get the total number of SIMs a user holds on ALL networks combined
    sims_all_networks = df_month_nid.groupby(["id_number"], observed=True).size()
    
    exactly_one_all = (sims_all_networks == 1).sum()
    two_to_ten_all = ((sims_all_networks > 1) & (sims_all_networks <= 10)).sum()
    wastage = (sims_all_networks > 10).sum()

    print("MULTI SIM OWNERSHIP AUDIT (SIMs held on ALL Networks):")
    print(f" * Exactly 1 SIM (Desired User):  {exactly_one_all:>12,}")
    print(f" * 2 to 10 SIMs (Power User):     {two_to_ten_all:>12,}")
    print(f" * >10 SIMs (Resource Wastage!):  {wastage:>12,} ")
    print("="*70)

Total MSISDNs Registered with NID:        65,723,549
Total Unique Subscribers (NID):     13,381,204
Average SIMs per Person:                        4.91
----------------------------------------------------------------------
NETWORK LOYALTY (Cross-Network Presence):
 * Active on exactly 1 Network:           7,521,799 (56.2%)
 * Active on 2+ Networks:                 5,859,405 (43.8%)
----------------------------------------------------------------------
COMPLIANCE AUDIT (SIMs held on a SINGLE Network):
 * Exactly 1 SIM (Standard User):         2,971,392
 * 2 to 10 SIMs (Power User):             9,644,193
 * >10 SIMs (UCC RULE VIOLATORS!):          765,619 
----------------------------------------------------------------------
MULTI SIM OWNERSHIP AUDIT (SIMs held on ALL Networks):
 * Exactly 1 SIM (Desired User):     2,458,458
 * 2 to 10 SIMs (Power User):        9,591,563
 * >10 SIMs (Resource Wastage!):     1,331,183 


Total MSISDNs Registered with NID:        36,870,973
Total Unique Subscribers (NID):      8,608,200
Average SIMs per Person:                        4.28
----------------------------------------------------------------------
NETWORK LOYALTY (Cross-Network Presence):
 * Active on exactly 1 Network:           6,392,607 (74.3%)
 * Active on 2+ Networks:                 2,215,593 (25.7%)
----------------------------------------------------------------------
COMPLIANCE AUDIT (SIMs held on a SINGLE Network):
 * Exactly 1 SIM (Standard User):         2,402,760
 * 2 to 10 SIMs (Power User):             5,727,604
 * >10 SIMs (UCC RULE VIOLATORS!):          477,836 
----------------------------------------------------------------------
MULTI SIM OWNERSHIP AUDIT (SIMs held on ALL Networks):
 * Exactly 1 SIM (Desired User):     2,213,736
 * 2 to 10 SIMs (Power User):        5,719,851
 * >10 SIMs (Resource Wastage!):       674,613 


Total MSISDNs Registered with NID:        35,319,130
Total Unique Subscribers (NID):      8,279,730
Average SIMs per Person:                        4.27
----------------------------------------------------------------------
NETWORK LOYALTY (Cross-Network Presence):
 * Active on exactly 1 Network:           6,194,711 (74.8%)
 * Active on 2+ Networks:                 2,085,019 (25.2%)
----------------------------------------------------------------------
COMPLIANCE AUDIT (SIMs held on a SINGLE Network):
 * Exactly 1 SIM (Standard User):         2,297,816
 * 2 to 10 SIMs (Power User):             5,533,411
 * >10 SIMs (UCC RULE VIOLATORS!):          448,503 
----------------------------------------------------------------------
MULTI SIM OWNERSHIP AUDIT (SIMs held on ALL Networks):
 * Exactly 1 SIM (Desired User):     2,125,525
 * 2 to 10 SIMs (Power User):        5,514,777
 * >10 SIMs (Resource Wastage!):       639,428 


Total MSISDNs Registered with NID:        25,813,912
Total Unique Subscribers (NID):      7,262,181
Average SIMs per Person:                        3.55
----------------------------------------------------------------------
NETWORK LOYALTY (Cross-Network Presence):
 * Active on exactly 1 Network:           5,696,753 (78.4%)
 * Active on 2+ Networks:                 1,565,428 (21.6%)
----------------------------------------------------------------------
COMPLIANCE AUDIT (SIMs held on a SINGLE Network):
 * Exactly 1 SIM (Standard User):         2,412,204
 * 2 to 10 SIMs (Power User):             4,576,889
 * >10 SIMs (UCC RULE VIOLATORS!):          273,088 
----------------------------------------------------------------------
MULTI SIM OWNERSHIP AUDIT (SIMs held on ALL Networks):
 * Exactly 1 SIM (Desired User):     2,237,117
 * 2 to 10 SIMs (Power User):        4,666,861
 * >10 SIMs (Resource Wastage!):       358,203 


Total MSISDNs Registered with NID:        28,657,581
Total Unique Subscribers (NID):      7,410,240
Average SIMs per Person:                        3.87
----------------------------------------------------------------------
NETWORK LOYALTY (Cross-Network Presence):
 * Active on exactly 1 Network:           5,828,838 (78.7%)
 * Active on 2+ Networks:                 1,581,402 (21.3%)
----------------------------------------------------------------------
COMPLIANCE AUDIT (SIMs held on a SINGLE Network):
 * Exactly 1 SIM (Standard User):         2,434,348
 * 2 to 10 SIMs (Power User):             4,579,019
 * >10 SIMs (UCC RULE VIOLATORS!):          396,873 
----------------------------------------------------------------------
MULTI SIM OWNERSHIP AUDIT (SIMs held on ALL Networks):
 * Exactly 1 SIM (Desired User):     2,268,606
 * 2 to 10 SIMs (Power User):        4,656,733
 * >10 SIMs (Resource Wastage!):       484,901 


Total MSISDNs Registered with NID:        22,055,804
Total Unique Subscribers (NID):      6,378,423
Average SIMs per Person:                        3.46
----------------------------------------------------------------------
NETWORK LOYALTY (Cross-Network Presence):
 * Active on exactly 1 Network:           5,212,464 (81.7%)
 * Active on 2+ Networks:                 1,165,959 (18.3%)
----------------------------------------------------------------------
COMPLIANCE AUDIT (SIMs held on a SINGLE Network):
 * Exactly 1 SIM (Standard User):         2,280,941
 * 2 to 10 SIMs (Power User):             3,833,807
 * >10 SIMs (UCC RULE VIOLATORS!):          263,675 
----------------------------------------------------------------------
MULTI SIM OWNERSHIP AUDIT (SIMs held on ALL Networks):
 * Exactly 1 SIM (Desired User):     2,147,238
 * 2 to 10 SIMs (Power User):        3,913,697
 * >10 SIMs (Resource Wastage!):       317,488 


Total MSISDNs Registered with NID:         4,403,847
Total Unique Subscribers (NID):      1,653,315
Average SIMs per Person:                        2.66
----------------------------------------------------------------------
NETWORK LOYALTY (Cross-Network Presence):
 * Active on exactly 1 Network:           1,646,335 (99.6%)
 * Active on 2+ Networks:                     6,980 (0.4%)
----------------------------------------------------------------------
COMPLIANCE AUDIT (SIMs held on a SINGLE Network):
 * Exactly 1 SIM (Standard User):           239,472
 * 2 to 10 SIMs (Power User):             1,405,404
 * >10 SIMs (UCC RULE VIOLATORS!):            8,439 
----------------------------------------------------------------------
MULTI SIM OWNERSHIP AUDIT (SIMs held on ALL Networks):
 * Exactly 1 SIM (Desired User):       239,282
 * 2 to 10 SIMs (Power User):        1,405,509
 * >10 SIMs (Resource Wastage!):         8,524 
